### Final Dataset


In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
#read the datasets needed
#these were all geopackages of the Hex Edits 
gdf1 = gpd.read_file("")
gdf2 = gpd.read_file("")
gdf3 = gpd.read_file("")
gdf4 = gpd.read_file("")
gdf5 = gpd.read_file("")
gdf6 = gpd.read_file("")


In [3]:
#read only the columns we want
cols1 = ["h3_ID", "geometry", "Total", "pa_acres", "dam_acres", "event_count", "heat_avg_extreme", "sea_acres", "flood_acres", "wildfire_avg"]
cols2 = ["h3_ID", "Non_Geo_Risk", "Non_Geo_Normalized"]
cols3 = ["h3_ID", "Non_Geo_CL", "Non_Geo_CL_Norm"]
cols4 = ["h3_ID", "Highly_Dependent", "Highly_Dependent_Index"]
cols5 = ["h3_ID", "Commercial Facilities", "Communications", "Dams", "Emergency Services", "Energy", "Financial",
         "Food & Agriculture", "Government Facilities", "Healthcare & Public Health", "Nuclear", "Transportation",
         "Water & Wastewater", "total", "Total_Index"]
cols6 = ["h3_ID", "Communications", "Food, Hydration, Shelter", "Safety and Security", "Water Systems",
         "Health and Medical", "Transportation", "Energy (Power & Fuel)", "Hazardous Materials", "total", "Total_Index"]

gdf1_sub = gdf1[cols1]
gdf2_sub = gdf2[cols2]
gdf3_sub = gdf3[cols3]
gdf4_sub = gdf4[cols4] 
gdf5_sub = gdf5[cols5]
gdf6_sub = gdf6[cols6]

In [ ]:
#rename columsn that need to fit shapefile length
#Subjective and up to you - 
gdf1_sub.rename(columns={
    "Total": "Total1",
    "event_count": "events",
    "heavt_avg_extreme": "heat_avg",
    "flood_acres": "fl_acres",
    "wildfire_avg": "fire_avg"
}, inplace=True)

gdf2_sub.rename(columns={
    "Non_Geo_Risk": "NonGeoRisk",
    "Non_Geo_Normalized": "NonGeoNorm"
}, inplace=True)

gdf3_sub.rename(columns={
    "Non_Geo_CL": "CL_NG_Risk",
    "Non_Geo_CL_Norm": "CL_NG_Norm"
}, inplace=True)

gdf4_sub.rename(columns={
    "Highly_Dependent": "High_D",
    "Highly_Dependent_Index": "HD_Index"
}, inplace=True)

gdf5_sub.rename(columns={
    "Commercial Facilities": "CIS_1",
    "Communications": "CIS_2",
    "Dams": "CIS_3",
    "Emergency Services": "CIS_4",
    "Energy": "CIS_5",
    "Financial": "CIS_6",
    "Food & Agriculture": "CIS_7",
    "Government Facilities": "CIS_8",
    "Healthcare & Public Health": "CIS_9",
    "Nuclear": "CIS_10",
    "Transportation": "CIS_11", 
    "Water & Wastewater": "CIS_12",
    "total": "CIS_Total",
    "Total_Index": "CIS_Norm"
}, inplace=True)

gdf6_sub.rename(columns={
    "Communications": "CL_1",
    "Energy (Power & Fuel)": "CL_2",
    "Food, Hydration, Shelter": "CL_3",
    "Hazardous Materials": "CL_4",
    "Health and Medical": "CL_5",
    "Safety and Security": "CL_6",
    "Transportation": "CL_7",
    "Water Systems": "CL_8",
    "total": "CL_Total",
    "Total_Index": "CL_Norm"
}, inplace=True)


In [5]:
merged = gdf1_sub.merge(gdf2_sub, on="h3_ID", how="left")
merged = merged.merge(gdf3_sub, on="h3_ID", how="left")
merged = merged.merge(gdf4_sub, on="h3_ID", how="left")
merged = merged.merge(gdf5_sub, on="h3_ID", how="left")
merged = merged.merge(gdf6_sub, on="h3_ID", how="left")

In [6]:
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=gdf1.crs)

In [7]:
#add column for total geographic and non-geographic for CISA and CL
merged["Risk_CL"] = merged["Total1"] + merged["CL_NG_Risk"]
merged["Risk_CISA"] = merged["Total1"] + merged["NonGeoRisk"]

#add column for total infrastructure, where Highly Dependent gets extra counts - 
merged["CL_TOT_HD"] = merged["High_D"] + merged["CL_Total"]
merged["CIS_TOT_HD"] = merged["High_D"] + merged["CIS_Total"]

In [ ]:
#save to H3 Edits
merged.to_file("Final_Data.gpkg", driver="GPKG")

In [ ]:
#making a shapefile as well, place in H3 Edits
merged.to_file("Final_Data.shp", driver="ESRI Shapefile")